In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DATA_PATH = "/content/drive/My Drive/Deep Learning Model/data_feat.csv"
OUTPUT_DIR = "/content/drive/My Drive/Deep Learning Model/"

DATE_COL  = "dt"
ID_COL    = "corp_tkr"
CLOSE_COL = "close"

CATEGORY_COLS = ["ml_industry_lvl_2", "ml_industry_lvl_3", "ml_industry_lvl_4"]  #@param

HORIZON = 1
TRAIN_WINDOW = 252
TEST_WINDOW = 10

CORR_THRESHOLD = 0.98
USE_ROBUST_SCALER = True
USE_CS_ZSCORE = True

FILLNA_VALUE = 0.0
MIN_TRAIN_ROWS = 5000
RANDOM_SEED = 42


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import RobustScaler
from tqdm.auto import tqdm
import os

np.random.seed(RANDOM_SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)


def drop_high_corr_features(X, threshold):
    if X.shape[1] <= 1:
        return X.columns.tolist(), []
    corr = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_cols = [c for c in upper.columns if (upper[c] >= threshold).any()]
    keep_cols = [c for c in X.columns if c not in drop_cols]
    return keep_cols, drop_cols


def cs_zscore_by_date(X, date_series):
    """
    Apply cross-sectional z-score normalization by date.

    For each trading day, features are standardized across the cross-section.

    Parameters
    ----------
    X : pd.DataFrame
        Feature matrix.
    date_series : pd.Series
        Series of dates aligned with X.

    Returns
    -------
    pd.DataFrame
        Cross-sectionally normalized feature matrix.
    """
    X = X.copy()
    X["_dt_tmp_"] = date_series.values
    for col in X.columns:
        if col == "_dt_tmp_":
            continue
        X[col] = X.groupby("_dt_tmp_")[col].transform(
            lambda s: (s - s.mean()) / (s.std(ddof=0) + 1e-6)
        )
    return X.drop(columns="_dt_tmp_")


def one_hot_align(train_df, test_df, cat_cols):
    train_oh = pd.get_dummies(train_df, columns=cat_cols, drop_first=False)
    test_oh  = pd.get_dummies(test_df,  columns=cat_cols, drop_first=False)
    test_oh = test_oh.reindex(columns=train_oh.columns, fill_value=0)

    return train_oh, test_oh

def sanitize_matrix(X: pd.DataFrame, fill=0.0, clip_abs=1e6) -> pd.DataFrame:
    """
    Sanitize feature matrix to improve numerical stability.
    This function handles:
    - inf / -inf values
    - extremely large values caused by numerical explosions
    - accidental non-numeric (object) columns
    """
    X = X.copy()
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.clip(lower=-clip_abs, upper=clip_abs)
    X = X.fillna(fill)
    return X.astype(np.float32)



In [ ]:
df=pd.read_csv(DATA_PATH, parse_dates=[DATE_COL])
cols_with_ret = [c for c in df.columns if "ret" in c.lower()]

print("❌ Columns to be dropped (contain 'ret'):")
print(cols_with_ret)

df = df.drop(columns=cols_with_ret)


In [ ]:
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values([ID_COL, DATE_COL]).reset_index(drop=True)

# label
label_col = f"ret_fwd_{HORIZON}"
future_close = df.groupby(ID_COL)[CLOSE_COL].shift(-HORIZON)
df[label_col] = future_close / df[CLOSE_COL] - 1
df = df.dropna(subset=[label_col]).reset_index(drop=True)

# numerical columns
exclude = {
    DATE_COL, ID_COL, CLOSE_COL, label_col,
    "equity_name", "country", "fs_entity_id", "renamed_ticker",
}
exclude |= {c for c in df.columns if "unnamed" in c.lower()}

num_cols = df.select_dtypes(include=[np.number]).columns
num_features = [c for c in num_cols if c not in exclude]

all_dates = df[DATE_COL].sort_values().unique()

pred_rows, meta_rows, coef_rows = [], [], []

starts = range(TRAIN_WINDOW, len(all_dates) - TEST_WINDOW + 1, TEST_WINDOW)

for split_id, start in enumerate(tqdm(starts), start=1):

    train_dates = all_dates[start - TRAIN_WINDOW:start]
    test_dates  = all_dates[start:start + TEST_WINDOW]

    train_df = df[df[DATE_COL].isin(train_dates)]
    test_df  = df[df[DATE_COL].isin(test_dates)]

    if len(train_df) < MIN_TRAIN_ROWS:
        continue

    # getting numerical & categorical columns
    X_train = train_df[num_features + CATEGORY_COLS].copy()
    X_test  = test_df[num_features + CATEGORY_COLS].copy()
    y_train = train_df[label_col].values

    X_train = X_train.fillna(FILLNA_VALUE)
    X_test  = X_test.fillna(FILLNA_VALUE)

    # changing into one hot
    X_train, X_test = one_hot_align(X_train, X_test, CATEGORY_COLS)

    # sanitize data
    X_train = sanitize_matrix(X_train, fill=FILLNA_VALUE, clip_abs=1e6)
    X_test  = sanitize_matrix(X_test,  fill=FILLNA_VALUE, clip_abs=1e6)

    # corr-drop
    keep_cols, dropped = drop_high_corr_features(X_train, CORR_THRESHOLD)
    X_train = X_train[keep_cols]
    X_test  = X_test[keep_cols]

    # scaler（only fit train)
    if USE_ROBUST_SCALER:
        scaler = RobustScaler()
        scaler.fit(X_train)
        X_train[:] = scaler.transform(X_train)
        X_test[:]  = scaler.transform(X_test)

    # z-score
    if USE_CS_ZSCORE:
        X_train = cs_zscore_by_date(X_train, train_df[DATE_COL])
        X_test  = cs_zscore_by_date(X_test,  test_df[DATE_COL])

    # Linear Regression
    model = LinearRegression()
    model.fit(X_train.values, y_train)
    preds = model.predict(X_test.values)

    out = test_df[[DATE_COL, ID_COL, label_col]].copy()
    out[f"pred_{HORIZON}"] = preds
    out["split_id"] = split_id
    pred_rows.append(out)

    meta_rows.append({
        "split_id": split_id,
        "train_start": train_dates[0],
        "train_end": train_dates[-1],
        "test_start": test_dates[0],
        "test_end": test_dates[-1],
        "n_features_after_oh": X_train.shape[1],
        "n_dropped_corr": len(dropped),
    })

    for name, coef in zip(X_train.columns, model.coef_):
        coef_rows.append({
            "split_id": split_id,
            "feature": name,
            "coef": coef,
        })


In [ ]:
result_df = pd.concat(pred_rows, ignore_index=True)

pred_cols = [
    DATE_COL,
    ID_COL,
    f"ret_fwd_{HORIZON}",
    f"pred_{HORIZON}",
    "split_id",
]

print(result_df[pred_cols].head())

# save predictions.csv
result_path = OUTPUT_DIR + "/predictions_1.csv"
result_df[pred_cols].to_csv(result_path, index=False)

print(f"prediction result saved to: {result_path}")

# prediction
pred = pd.read_csv(result_path)
pred["dt"] = pd.to_datetime(pred["dt"])

# merge close
pred = pred.merge(
    df[["dt", "corp_tkr", "close"]],
    on=["dt", "corp_tkr"],
    how="left"
)

# check
assert pred["close"].notna().all(), "Some close values missing after merge"

# continue using pred
pred.to_csv(result_path, index=False)


In [ ]:
# print split Top coefficient
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coef": model.coef_,
})

coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)

print(f"\n===== Split {split_id} | Top 10 coefficients =====")
print(coef_df[["feature", "coef"]].head(10))


In [ ]:
coef_all = pd.DataFrame(coef_rows)


In [ ]:
coef_summary = (
    coef_all
    .groupby("feature")["coef"]
    .agg(
        mean_coef="mean",
        std_coef="std",
        pos_ratio=lambda x: (x > 0).mean(),
        neg_ratio=lambda x: (x < 0).mean(),
        n_obs="count",
    )
    .reset_index()
)

# according to |mean_coef| rank，most important features
coef_summary["abs_mean"] = coef_summary["mean_coef"].abs()
coef_summary = coef_summary.sort_values("abs_mean", ascending=False)

print("\n===== Coefficient summary (top 15) =====")
print(coef_summary.head(15))
